In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import wilcoxon

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_predict,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from tqdm.auto import tqdm


warnings.filterwarnings("ignore")


DATA_PATH = Path(
    "./data/voc_dataset_1+2_vs_3.mat"
)

OUTPUT_DIR = Path(
    "./result/logistic_refinement"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


TOP_K_LIST = [150, 200, 250, 300]

PENALTY_LIST = ["l1", "l2"]

CLASS_WEIGHT_LIST = [
    None,
    "balanced",
]

C_VALUES = [
    0.001,
    0.01,
    0.1,
    1.0,
    10.0,
    100.0,
]


OUTER_REPEATS = 30
TEST_SIZE = 0.20
INNER_FOLDS = 5
SEED = 42


print("数据文件：", DATA_PATH.resolve())
print("结果目录：", OUTPUT_DIR.resolve())
print("Top-K：", TOP_K_LIST)
print("Penalty：", PENALTY_LIST)
print("Class weight：", CLASS_WEIGHT_LIST)
print("C：", C_VALUES)
print("外层重复次数：", OUTER_REPEATS)

数据文件： /home/liuxy/a-projects/BPD_jj/data/voc_dataset_1+2_vs_3.mat
结果目录： /home/liuxy/a-projects/BPD_jj/result/logistic_refinement
Top-K： [150, 200, 250, 300]
Penalty： ['l1', 'l2']
Class weight： [None, 'balanced']
C： [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
外层重复次数： 30


In [2]:
def decode_matlab_string(value):
    while (
        isinstance(value, np.ndarray)
        and value.size == 1
    ):
        value = value.reshape(-1)[0]

    if isinstance(value, bytes):
        return value.decode(
            "utf-8",
            errors="replace",
        ).strip()

    if isinstance(value, np.ndarray):
        if value.dtype.kind in {"U", "S"}:
            return "".join(
                value.astype(str).reshape(-1)
            ).strip()

        return str(value.squeeze()).strip()

    return str(value).strip()


if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"找不到数据文件：{DATA_PATH.resolve()}"
    )


mat_data = sio.loadmat(DATA_PATH)

X = np.asarray(
    mat_data["X"],
    dtype=np.float64,
)

y = np.asarray(
    mat_data["y"],
    dtype=np.int64,
).reshape(-1)


if "feat_names" in mat_data:
    feature_names = [
        decode_matlab_string(value)
        for value in mat_data[
            "feat_names"
        ].reshape(-1)
    ]
else:
    feature_names = [
        f"VOC_{index}"
        for index in range(X.shape[1])
    ]


if len(feature_names) != X.shape[1]:
    feature_names = [
        f"VOC_{index}"
        for index in range(X.shape[1])
    ]


if X.ndim != 2:
    raise ValueError(
        f"X 应为二维矩阵，当前形状：{X.shape}"
    )

if len(y) != X.shape[0]:
    raise ValueError(
        "X 样本数和 y 标签数不一致。"
    )

if not np.isfinite(X).all():
    raise ValueError(
        "数据中存在 NaN 或 Inf。"
    )


print("数据读取完成")
print("-" * 60)
print("样本数：", X.shape[0])
print("特征数：", X.shape[1])
print(
    "类别数量：",
    dict(
        zip(
            *np.unique(
                y,
                return_counts=True,
            )
        )
    ),
)

数据读取完成
------------------------------------------------------------
样本数： 159
特征数： 780
类别数量： {np.int64(0): np.int64(106), np.int64(1): np.int64(53)}


In [3]:
def calculate_metrics(
    y_true,
    y_pred,
    scores,
):
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    )

    tn, fp, fn, tp = cm.ravel()

    eps = 1e-12

    sensitivity = tp / (
        tp + fn + eps
    )

    specificity = tn / (
        tn + fp + eps
    )

    ppv = tp / (
        tp + fp + eps
    )

    npv = tn / (
        tn + fn + eps
    )

    try:
        auc_value = roc_auc_score(
            y_true,
            scores,
        )
    except ValueError:
        auc_value = np.nan

    return {
        "Sensitivity": float(sensitivity),
        "Specificity": float(specificity),
        "PPV": float(ppv),
        "NPV": float(npv),

        "Accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "Balanced_Accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "F1": float(
            f1_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),

        "AUC": float(auc_value),
    }


def select_threshold(
    y_true,
    probabilities,
):
    thresholds = np.linspace(
        0.05,
        0.95,
        181,
    )

    best_result = None

    for threshold in thresholds:
        predictions = (
            probabilities >= threshold
        ).astype(int)

        accuracy = accuracy_score(
            y_true,
            predictions,
        )

        f1 = f1_score(
            y_true,
            predictions,
            zero_division=0,
        )

        balanced_accuracy = (
            balanced_accuracy_score(
                y_true,
                predictions,
            )
        )

        joint_score = min(
            accuracy,
            f1,
        )

        secondary_score = (
            accuracy
            + f1
            + balanced_accuracy
        ) / 3.0

        candidate = {
            "Threshold": float(threshold),
            "Accuracy": float(accuracy),
            "F1": float(f1),
            "Joint": float(joint_score),
            "Secondary": float(
                secondary_score
            ),
        }

        if best_result is None:
            best_result = candidate
            continue

        if (
            candidate["Joint"]
            > best_result["Joint"]
        ):
            best_result = candidate

        elif np.isclose(
            candidate["Joint"],
            best_result["Joint"],
        ):
            if (
                candidate["Secondary"]
                > best_result["Secondary"]
            ):
                best_result = candidate

    return best_result


def mean_ci(values):
    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
    ]

    mean_value = values.mean()
    std_value = values.std(ddof=1)

    sem_value = (
        std_value / np.sqrt(len(values))
    )

    t_value = stats.t.ppf(
        0.975,
        df=len(values) - 1,
    )

    lower = (
        mean_value
        - t_value * sem_value
    )

    upper = (
        mean_value
        + t_value * sem_value
    )

    return (
        float(mean_value),
        float(std_value),
        float(lower),
        float(upper),
    )

In [4]:
TEST_REPEATS = 2

test_rows = []

all_indices = np.arange(
    X.shape[0]
)


for repeat_index in tqdm(
    range(TEST_REPEATS),
    desc="Smoke test",
):
    outer_seed = (
        SEED
        + repeat_index * 1000
    )

    train_indices, test_indices = train_test_split(
        all_indices,
        test_size=TEST_SIZE,
        random_state=outer_seed,
        stratify=y,
    )

    X_train = X[train_indices]
    y_train = y[train_indices]

    X_test = X[test_indices]
    y_test = y[test_indices]


    inner_cv = StratifiedKFold(
        n_splits=INNER_FOLDS,
        shuffle=True,
        random_state=outer_seed + 1,
    )


    pipeline = Pipeline(
        steps=[
            (
                "feature_selection",
                SelectKBest(
                    score_func=f_classif,
                ),
            ),

            (
                "scaler",
                StandardScaler(),
            ),

            (
                "classifier",
                LogisticRegression(
                    solver="liblinear",
                    max_iter=10000,
                    random_state=outer_seed,
                ),
            ),
        ]
    )


    parameter_grid = {
        "feature_selection__k":
            TOP_K_LIST,

        "classifier__penalty":
            PENALTY_LIST,

        "classifier__class_weight":
            CLASS_WEIGHT_LIST,

        "classifier__C":
            C_VALUES,
    }


    search = GridSearchCV(
        estimator=pipeline,
        param_grid=parameter_grid,
        scoring="f1",
        cv=inner_cv,
        n_jobs=-1,
        refit=True,
        error_score="raise",
    )


    search.fit(
        X_train,
        y_train,
    )


    best_model = search.best_estimator_


    oof_probabilities = cross_val_predict(
        best_model,
        X_train,
        y_train,
        cv=inner_cv,
        method="predict_proba",
        n_jobs=-1,
    )[:, 1]


    threshold_result = select_threshold(
        y_true=y_train,
        probabilities=oof_probabilities,
    )


    test_probabilities = (
        best_model.predict_proba(
            X_test
        )[:, 1]
    )


    test_predictions = (
        test_probabilities
        >= threshold_result["Threshold"]
    ).astype(int)


    metrics = calculate_metrics(
        y_true=y_test,
        y_pred=test_predictions,
        scores=test_probabilities,
    )


    row = {
        "Repeat": repeat_index,
        "Top_K": search.best_params_[
            "feature_selection__k"
        ],
        "Penalty": search.best_params_[
            "classifier__penalty"
        ],
        "Class_Weight": str(
            search.best_params_[
                "classifier__class_weight"
            ]
        ),
        "C": search.best_params_[
            "classifier__C"
        ],
        "Threshold":
            threshold_result["Threshold"],
    }

    row.update(metrics)

    test_rows.append(row)


test_df = pd.DataFrame(test_rows)

display(test_df)

print("小规模测试完成。")

Smoke test:   0%|          | 0/2 [00:00<?, ?it/s]

,Repeat,Top_K,Penalty,Class_Weight,C,Threshold,Sensitivity,Specificity,PPV,NPV,Accuracy,Balanced_Accuracy,F1,AUC
0,0,300,l2,None,0.10,0.520,0.636364,0.952381,0.875000,0.833333,0.84375,0.794372,0.736842,0.857143
1,1,250,l2,balanced,0.01,0.485,0.636364,0.714286,0.538462,0.789474,0.68750,0.675325,0.583333,0.813853


小规模测试完成。


In [5]:
# ============================================================
# 正式 Logistic Regression 精细化实验
#
# 搜索：
# Top-K = 150, 200, 250, 300
# penalty = l1, l2
# class_weight = None, balanced
# C = 0.001, 0.01, 0.1, 1, 10, 100
#
# 外层重复：30 次
# ============================================================

FORMAL_REPEATS = 30

FORMAL_RESULT_PATH = (
    OUTPUT_DIR
    / "logistic_refinement_per_repeat.csv"
)

all_indices = np.arange(
    X.shape[0]
)

formal_rows = []


progress = tqdm(
    range(FORMAL_REPEATS),
    desc="Logistic refinement",
)


for repeat_index in progress:

    # 与之前正式实验保持一致的外层种子
    outer_seed = (
        SEED
        + repeat_index * 1000
    )


    # --------------------------------------------------------
    # 外层训练集和测试集
    # --------------------------------------------------------

    train_indices, test_indices = train_test_split(
        all_indices,
        test_size=TEST_SIZE,
        random_state=outer_seed,
        stratify=y,
    )


    X_train = X[
        train_indices
    ]

    y_train = y[
        train_indices
    ]

    X_test = X[
        test_indices
    ]

    y_test = y[
        test_indices
    ]


    # --------------------------------------------------------
    # 内层交叉验证：选择模型参数
    # --------------------------------------------------------

    parameter_cv = StratifiedKFold(
        n_splits=INNER_FOLDS,
        shuffle=True,
        random_state=outer_seed + 1,
    )


    pipeline = Pipeline(
        steps=[
            (
                "feature_selection",
                SelectKBest(
                    score_func=f_classif,
                ),
            ),

            (
                "scaler",
                StandardScaler(),
            ),

            (
                "classifier",
                LogisticRegression(
                    solver="liblinear",
                    max_iter=10000,
                    random_state=outer_seed,
                ),
            ),
        ]
    )


    parameter_grid = {
        "feature_selection__k":
            TOP_K_LIST,

        "classifier__penalty":
            PENALTY_LIST,

        "classifier__class_weight":
            CLASS_WEIGHT_LIST,

        "classifier__C":
            C_VALUES,
    }


    search = GridSearchCV(
        estimator=pipeline,
        param_grid=parameter_grid,
        scoring="f1",
        cv=parameter_cv,
        n_jobs=-1,
        refit=True,
        error_score="raise",
        return_train_score=False,
    )


    search.fit(
        X_train,
        y_train,
    )


    best_model = (
        search.best_estimator_
    )


    # --------------------------------------------------------
    # 使用另一组训练集交叉验证预测选择阈值
    # 测试集不参与阈值选择
    # --------------------------------------------------------

    threshold_cv = StratifiedKFold(
        n_splits=INNER_FOLDS,
        shuffle=True,
        random_state=outer_seed + 2,
    )


    oof_probabilities = cross_val_predict(
        best_model,
        X_train,
        y_train,
        cv=threshold_cv,
        method="predict_proba",
        n_jobs=-1,
    )[:, 1]


    threshold_result = select_threshold(
        y_true=y_train,
        probabilities=oof_probabilities,
    )


    best_threshold = (
        threshold_result[
            "Threshold"
        ]
    )


    # --------------------------------------------------------
    # 独立外层测试集评估
    # --------------------------------------------------------

    test_probabilities = (
        best_model.predict_proba(
            X_test
        )[:, 1]
    )


    test_predictions = (
        test_probabilities
        >= best_threshold
    ).astype(int)


    metrics = calculate_metrics(
        y_true=y_test,
        y_pred=test_predictions,
        scores=test_probabilities,
    )


    # --------------------------------------------------------
    # 记录最终选择的特征数量和有效非零系数数量
    # --------------------------------------------------------

    selector = best_model.named_steps[
        "feature_selection"
    ]

    classifier = best_model.named_steps[
        "classifier"
    ]


    selected_indices = np.flatnonzero(
        selector.get_support()
    )


    coefficients = (
        classifier.coef_
        .reshape(-1)
    )


    nonzero_coefficients = int(
        np.sum(
            np.abs(coefficients)
            > 1e-8
        )
    )


    row = {
        "Repeat": repeat_index,
        "Outer_Seed": outer_seed,

        "Top_K": int(
            search.best_params_[
                "feature_selection__k"
            ]
        ),

        "Penalty": search.best_params_[
            "classifier__penalty"
        ],

        "Class_Weight": str(
            search.best_params_[
                "classifier__class_weight"
            ]
        ),

        "C": float(
            search.best_params_[
                "classifier__C"
            ]
        ),

        "Threshold": float(
            best_threshold
        ),

        "Inner_Best_F1": float(
            search.best_score_
        ),

        "Inner_Threshold_Accuracy": float(
            threshold_result[
                "Accuracy"
            ]
        ),

        "Inner_Threshold_F1": float(
            threshold_result[
                "F1"
            ]
        ),

        "Selected_Features": int(
            len(selected_indices)
        ),

        "Nonzero_Coefficients":
            nonzero_coefficients,
    }


    row.update(
        metrics
    )


    formal_rows.append(
        row
    )


    # 每完成一次就保存，防止中途断开后结果全部丢失
    pd.DataFrame(
        formal_rows
    ).to_csv(
        FORMAL_RESULT_PATH,
        index=False,
        encoding="utf-8-sig",
    )


    progress.set_postfix(
        {
            "repeat":
                f"{repeat_index + 1}/"
                f"{FORMAL_REPEATS}",

            "K": row["Top_K"],

            "penalty":
                row["Penalty"],

            "weight":
                row["Class_Weight"],

            "ACC":
                f"{metrics['Accuracy']:.3f}",

            "F1":
                f"{metrics['F1']:.3f}",

            "AUC":
                f"{metrics['AUC']:.3f}",
        }
    )


formal_df = pd.DataFrame(
    formal_rows
)


display(
    formal_df
)


print("=" * 70)
print("正式精细化实验完成")
print("结果文件：", FORMAL_RESULT_PATH.resolve())
print("=" * 70)

Logistic refinement:   0%|          | 0/30 [00:00<?, ?it/s]

,Repeat,Outer_Seed,Top_K,Penalty,Class_Weight,C,Threshold,Inner_Best_F1,Inner_Threshold_Accuracy,Inner_Threshold_F1,Selected_Features,Nonzero_Coefficients,Sensitivity,Specificity,PPV,NPV,Accuracy,Balanced_Accuracy,F1,AUC
0,0,42,300,l2,None,0.10,0.325,0.569108,0.637795,0.581818,300,300,0.636364,0.809524,0.636364,0.809524,0.75000,0.722944,0.636364,0.857143
1,1,1042,250,l2,balanced,0.01,0.520,0.664131,0.708661,0.574713,250,250,0.545455,0.761905,0.545455,0.761905,0.68750,0.653680,0.545455,0.813853
2,2,2042,200,l2,None,10.00,0.050,0.646536,0.645669,0.594595,200,200,0.818182,0.428571,0.428571,0.818182,0.56250,0.623377,0.562500,0.774892
3,3,3042,250,l1,balanced,1.00,0.185,0.622506,0.582677,0.582677,250,59,0.818182,0.714286,0.600000,0.882353,0.75000,0.766234,0.692308,0.870130
4,4,4042,300,l2,balanced,0.10,0.280,0.605556,0.637795,0.603448,300,300,0.727273,0.428571,0.400000,0.750000,0.53125,0.577922,0.516129,0.740260
5,5,5042,200,l2,balanced,1.00,0.650,0.662222,0.795276,0.690476,200,200,0.636364,0.761905,0.583333,0.800000,0.71875,0.699134,0.608696,0.735931
6,6,6042,150,l1,balanced,1.00,0.205,0.564009,0.645669,0.594595,150,37,0.727273,0.761905,0.615385,0.842105,0.75000,0.744589,0.666667,0.822511
7,7,7042,200,l2,None,0.10,0.345,0.666471,0.732283,0.673077,200,200,0.545455,0.619048,0.428571,0.722222,0.59375,0.582251,0.480000,0.753247
8,8,8042,150,l2,None,0.10,0.435,0.716457,0.755906,0.659341,150,150,0.545455,0.714286,0.500000,0.750000,0.65625,0.629870,0.521739,0.705628
9,9,9042,250,l2,balanced,0.10,0.565,0.694591,0.795276,0.697674,250,250,0.727273,0.666667,0.533333,0.823529,0.68750,0.696970,0.615385,0.679654


正式精细化实验完成
结果文件： /home/liuxy/a-projects/BPD_jj/result/logistic_refinement/logistic_refinement_per_repeat.csv


In [6]:
# ============================================================
# 正式精细化实验汇总
# ============================================================

formal_df = pd.read_csv(
    FORMAL_RESULT_PATH
)


metric_names = [
    "Sensitivity",
    "Specificity",
    "PPV",
    "NPV",
    "Accuracy",
    "Balanced_Accuracy",
    "F1",
    "AUC",
]


summary_rows = []


for metric_name in metric_names:

    (
        mean_value,
        std_value,
        lower,
        upper,
    ) = mean_ci(
        formal_df[
            metric_name
        ].values
    )


    summary_rows.append(
        {
            "Metric": metric_name,
            "Mean": mean_value,
            "Std": std_value,
            "CI95_Lower": lower,
            "CI95_Upper": upper,
        }
    )


refinement_summary_df = pd.DataFrame(
    summary_rows
)


REFINEMENT_SUMMARY_PATH = (
    OUTPUT_DIR
    / "logistic_refinement_summary.csv"
)


refinement_summary_df.to_csv(
    REFINEMENT_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)


display(
    refinement_summary_df
)


print(
    "汇总结果已保存：",
    REFINEMENT_SUMMARY_PATH.resolve(),
)

,Metric,Mean,Std,CI95_Lower,CI95_Upper
0,Sensitivity,0.630303,0.170382,0.566681,0.693925
1,Specificity,0.733333,0.120338,0.688398,0.778268
2,PPV,0.566440,0.127976,0.518653,0.614226
3,NPV,0.796575,0.066096,0.771894,0.821255
4,Accuracy,0.697917,0.077566,0.668953,0.726880
5,Balanced_Accuracy,0.681818,0.082659,0.650953,0.712683
6,F1,0.582298,0.120418,0.537333,0.627262
7,AUC,0.762771,0.077593,0.733797,0.791744


汇总结果已保存： /home/liuxy/a-projects/BPD_jj/result/logistic_refinement/logistic_refinement_summary.csv


In [7]:
# ============================================================
# 最佳参数选择频率
# ============================================================

configuration_frequency_df = (
    formal_df
    .groupby(
        [
            "Top_K",
            "Penalty",
            "Class_Weight",
            "C",
        ],
        dropna=False,
    )
    .size()
    .reset_index(
        name="Selection_Count"
    )
)


configuration_frequency_df[
    "Selection_Frequency"
] = (
    configuration_frequency_df[
        "Selection_Count"
    ]
    / len(formal_df)
)


configuration_frequency_df = (
    configuration_frequency_df
    .sort_values(
        by=[
            "Selection_Count",
            "Top_K",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


CONFIGURATION_PATH = (
    OUTPUT_DIR
    / "logistic_refinement_configuration_frequency.csv"
)


configuration_frequency_df.to_csv(
    CONFIGURATION_PATH,
    index=False,
    encoding="utf-8-sig",
)


display(
    configuration_frequency_df.head(20)
)


print(
    "参数频率文件已保存：",
    CONFIGURATION_PATH.resolve(),
)

,Top_K,Penalty,Class_Weight,C,Selection_Count,Selection_Frequency
0,250,l2,balanced,0.10,5,0.166667
1,300,l2,NaN,0.10,4,0.133333
2,250,l1,balanced,1.00,3,0.100000
3,150,l1,balanced,0.10,2,0.066667
4,150,l1,balanced,1.00,1,0.033333
5,150,l1,balanced,10.00,1,0.033333
6,150,l2,balanced,100.00,1,0.033333
7,150,l2,NaN,0.10,1,0.033333
8,200,l1,NaN,1.00,1,0.033333
9,200,l2,balanced,0.10,1,0.033333


参数频率文件已保存： /home/liuxy/a-projects/BPD_jj/result/logistic_refinement/logistic_refinement_configuration_frequency.csv


In [8]:
print("Top-K 选择次数")
display(
    formal_df[
        "Top_K"
    ].value_counts()
    .sort_index()
    .rename_axis("Top_K")
    .reset_index(
        name="Count"
    )
)


print("Penalty 选择次数")
display(
    formal_df[
        "Penalty"
    ].value_counts()
    .rename_axis("Penalty")
    .reset_index(
        name="Count"
    )
)


print("Class weight 选择次数")
display(
    formal_df[
        "Class_Weight"
    ].value_counts()
    .rename_axis("Class_Weight")
    .reset_index(
        name="Count"
    )
)


print("C 选择次数")
display(
    formal_df[
        "C"
    ].value_counts()
    .sort_index()
    .rename_axis("C")
    .reset_index(
        name="Count"
    )
)

Top-K 选择次数


,Top_K,Count
0,150,6
1,200,5
2,250,10
3,300,9


Penalty 选择次数


,Penalty,Count
0,l2,20
1,l1,10


Class weight 选择次数


,Class_Weight,Count
0,balanced,17


C 选择次数


,C,Count
0,0.01,2
1,0.10,16
2,1.00,8
3,10.00,2
4,100.00,2


In [9]:
# ============================================================
# 固定参数确认实验
# ============================================================

FIXED_CONFIGS = [
    {
        "Config": "A_Top200_L2_balanced",
        "Top_K": 200,
        "Penalty": "l2",
        "Class_Weight": "balanced",
        "C": 0.1,
    },
    {
        "Config": "B_Top250_L2_balanced",
        "Top_K": 250,
        "Penalty": "l2",
        "Class_Weight": "balanced",
        "C": 0.1,
    },
    {
        "Config": "C_Top300_L2_none",
        "Top_K": 300,
        "Penalty": "l2",
        "Class_Weight": None,
        "C": 0.1,
    },
]

CONFIRM_REPEATS = 30

CONFIRM_RESULT_PATH = (
    OUTPUT_DIR
    / "fixed_configuration_per_repeat.csv"
)

confirmation_rows = []

all_indices = np.arange(X.shape[0])

progress = tqdm(
    total=CONFIRM_REPEATS * len(FIXED_CONFIGS),
    desc="Fixed configuration confirmation",
)

for repeat_index in range(CONFIRM_REPEATS):

    outer_seed = SEED + repeat_index * 1000

    train_indices, test_indices = train_test_split(
        all_indices,
        test_size=TEST_SIZE,
        random_state=outer_seed,
        stratify=y,
    )

    X_train = X[train_indices]
    y_train = y[train_indices]

    X_test = X[test_indices]
    y_test = y[test_indices]

    for config in FIXED_CONFIGS:

        model = Pipeline(
            steps=[
                (
                    "feature_selection",
                    SelectKBest(
                        score_func=f_classif,
                        k=config["Top_K"],
                    ),
                ),
                (
                    "scaler",
                    StandardScaler(),
                ),
                (
                    "classifier",
                    LogisticRegression(
                        penalty=config["Penalty"],
                        class_weight=config["Class_Weight"],
                        C=config["C"],
                        solver="liblinear",
                        max_iter=10000,
                        random_state=outer_seed,
                    ),
                ),
            ]
        )

        threshold_cv = StratifiedKFold(
            n_splits=INNER_FOLDS,
            shuffle=True,
            random_state=outer_seed + 2,
        )

        # 只使用训练集交叉验证预测确定阈值
        oof_probabilities = cross_val_predict(
            model,
            X_train,
            y_train,
            cv=threshold_cv,
            method="predict_proba",
            n_jobs=-1,
        )[:, 1]

        threshold_result = select_threshold(
            y_true=y_train,
            probabilities=oof_probabilities,
        )

        best_threshold = threshold_result["Threshold"]

        # 在全部外层训练集上拟合
        model.fit(X_train, y_train)

        test_probabilities = model.predict_proba(
            X_test
        )[:, 1]

        test_predictions = (
            test_probabilities >= best_threshold
        ).astype(int)

        metrics = calculate_metrics(
            y_true=y_test,
            y_pred=test_predictions,
            scores=test_probabilities,
        )

        row = {
            "Repeat": repeat_index,
            "Outer_Seed": outer_seed,
            "Config": config["Config"],
            "Top_K": config["Top_K"],
            "Penalty": config["Penalty"],
            "Class_Weight": str(config["Class_Weight"]),
            "C": config["C"],
            "Threshold": best_threshold,
        }

        row.update(metrics)
        confirmation_rows.append(row)

        pd.DataFrame(
            confirmation_rows
        ).to_csv(
            CONFIRM_RESULT_PATH,
            index=False,
            encoding="utf-8-sig",
        )

        progress.update(1)
        progress.set_postfix(
            {
                "repeat": repeat_index + 1,
                "config": config["Config"],
                "ACC": f"{metrics['Accuracy']:.3f}",
                "F1": f"{metrics['F1']:.3f}",
                "AUC": f"{metrics['AUC']:.3f}",
            }
        )

progress.close()

confirmation_df = pd.DataFrame(
    confirmation_rows
)

print("固定参数确认实验完成。")
print("结果文件：", CONFIRM_RESULT_PATH.resolve())

display(confirmation_df.head())

Fixed configuration confirmation:   0%|          | 0/90 [00:00<?, ?it/s]

固定参数确认实验完成。
结果文件： /home/liuxy/a-projects/BPD_jj/result/logistic_refinement/fixed_configuration_per_repeat.csv


,Repeat,Outer_Seed,Config,Top_K,Penalty,Class_Weight,C,Threshold,Sensitivity,Specificity,PPV,NPV,Accuracy,Balanced_Accuracy,F1,AUC
0,0,42,A_Top200_L2_balanced,200,l2,balanced,0.1,0.540,0.636364,0.952381,0.875000,0.833333,0.84375,0.794372,0.736842,0.852814
1,0,42,B_Top250_L2_balanced,250,l2,balanced,0.1,0.505,0.636364,0.904762,0.777778,0.826087,0.81250,0.770563,0.700000,0.883117
2,0,42,C_Top300_L2_none,300,l2,None,0.1,0.325,0.636364,0.809524,0.636364,0.809524,0.75000,0.722944,0.636364,0.857143
3,1,1042,A_Top200_L2_balanced,200,l2,balanced,0.1,0.260,0.909091,0.619048,0.555556,0.928571,0.71875,0.764069,0.689655,0.878788
4,1,1042,B_Top250_L2_balanced,250,l2,balanced,0.1,0.255,0.909091,0.571429,0.526316,0.923077,0.68750,0.740260,0.666667,0.883117


In [10]:
# ============================================================
# 固定方案汇总
# ============================================================

confirmation_df = pd.read_csv(
    CONFIRM_RESULT_PATH
)

metric_names = [
    "Sensitivity",
    "Specificity",
    "PPV",
    "NPV",
    "Accuracy",
    "Balanced_Accuracy",
    "F1",
    "AUC",
]

fixed_summary_rows = []

for config_name, group_df in confirmation_df.groupby("Config"):

    row = {
        "Config": config_name,
        "Top_K": int(group_df["Top_K"].iloc[0]),
        "Penalty": group_df["Penalty"].iloc[0],
        "Class_Weight": group_df["Class_Weight"].iloc[0],
        "C": group_df["C"].iloc[0],
    }

    for metric_name in metric_names:

        mean_value, std_value, lower, upper = mean_ci(
            group_df[metric_name].values
        )

        row[f"{metric_name}_Mean"] = mean_value
        row[f"{metric_name}_Std"] = std_value
        row[f"{metric_name}_CI95_Lower"] = lower
        row[f"{metric_name}_CI95_Upper"] = upper

    row["Joint_Mean"] = min(
        row["Accuracy_Mean"],
        row["F1_Mean"],
    )

    fixed_summary_rows.append(row)

fixed_summary_df = pd.DataFrame(
    fixed_summary_rows
).sort_values(
    by=[
        "Joint_Mean",
        "F1_Mean",
        "Accuracy_Mean",
        "AUC_Mean",
    ],
    ascending=False,
).reset_index(drop=True)

FIXED_SUMMARY_PATH = (
    OUTPUT_DIR
    / "fixed_configuration_summary.csv"
)

fixed_summary_df.to_csv(
    FIXED_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

display(
    fixed_summary_df[
        [
            "Config",
            "Accuracy_Mean",
            "Accuracy_Std",
            "F1_Mean",
            "F1_Std",
            "AUC_Mean",
            "AUC_Std",
            "Sensitivity_Mean",
            "Specificity_Mean",
            "Joint_Mean",
        ]
    ]
)

print("汇总文件：", FIXED_SUMMARY_PATH.resolve())

,Config,Accuracy_Mean,Accuracy_Std,F1_Mean,F1_Std,AUC_Mean,AUC_Std,Sensitivity_Mean,Specificity_Mean,Joint_Mean
0,B_Top250_L2_balanced,0.697917,0.070278,0.594311,0.102529,0.774459,0.085811,0.660606,0.717460,0.594311
1,C_Top300_L2_none,0.690625,0.073563,0.583794,0.101755,0.777489,0.077695,0.642424,0.715873,0.583794
2,A_Top200_L2_balanced,0.689583,0.082856,0.583268,0.125603,0.764502,0.089238,0.651515,0.709524,0.583268


汇总文件： /home/liuxy/a-projects/BPD_jj/result/logistic_refinement/fixed_configuration_summary.csv


In [11]:
# ============================================================
# 固定方案两两配对检验
# ============================================================

from itertools import combinations

paired_rows = []

config_names = confirmation_df["Config"].unique()

for config_a, config_b in combinations(config_names, 2):

    df_a = confirmation_df[
        confirmation_df["Config"] == config_a
    ].sort_values("Repeat")

    df_b = confirmation_df[
        confirmation_df["Config"] == config_b
    ].sort_values("Repeat")

    for metric_name in [
        "Accuracy",
        "F1",
        "AUC",
    ]:

        values_a = df_a[metric_name].to_numpy(dtype=float)
        values_b = df_b[metric_name].to_numpy(dtype=float)

        try:
            statistic, p_value = wilcoxon(
                values_b,
                values_a,
                zero_method="wilcox",
                alternative="two-sided",
            )
        except ValueError:
            statistic = np.nan
            p_value = np.nan

        paired_rows.append(
            {
                "Config_A": config_a,
                "Config_B": config_b,
                "Metric": metric_name,
                "Mean_A": values_a.mean(),
                "Mean_B": values_b.mean(),
                "B_Minus_A": (
                    values_b - values_a
                ).mean(),
                "Wilcoxon_Statistic": statistic,
                "P_Value": p_value,
            }
        )

fixed_paired_df = pd.DataFrame(
    paired_rows
)

FIXED_PAIRED_PATH = (
    OUTPUT_DIR
    / "fixed_configuration_paired_test.csv"
)

fixed_paired_df.to_csv(
    FIXED_PAIRED_PATH,
    index=False,
    encoding="utf-8-sig",
)

display(fixed_paired_df)

print("配对检验文件：", FIXED_PAIRED_PATH.resolve())

,Config_A,Config_B,Metric,Mean_A,Mean_B,B_Minus_A,Wilcoxon_Statistic,P_Value
0,A_Top200_L2_balanced,B_Top250_L2_balanced,Accuracy,0.689583,0.697917,0.008333,91.0,0.383651
1,A_Top200_L2_balanced,B_Top250_L2_balanced,F1,0.583268,0.594311,0.011043,154.0,0.264492
2,A_Top200_L2_balanced,B_Top250_L2_balanced,AUC,0.764502,0.774459,0.009957,145.5,0.119112
3,A_Top200_L2_balanced,C_Top300_L2_none,Accuracy,0.689583,0.690625,0.001042,145.0,0.884822
4,A_Top200_L2_balanced,C_Top300_L2_none,F1,0.583268,0.583794,0.000527,180.5,0.838182
5,A_Top200_L2_balanced,C_Top300_L2_none,AUC,0.764502,0.777489,0.012987,147.5,0.206194
6,B_Top250_L2_balanced,C_Top300_L2_none,Accuracy,0.697917,0.690625,-0.007292,113.5,0.443259
7,B_Top250_L2_balanced,C_Top300_L2_none,F1,0.594311,0.583794,-0.010516,157.5,0.300124
8,B_Top250_L2_balanced,C_Top300_L2_none,AUC,0.774459,0.777489,0.003030,167.0,0.596773


配对检验文件： /home/liuxy/a-projects/BPD_jj/result/logistic_refinement/fixed_configuration_paired_test.csv


In [12]:
# ============================================================
# 最终 Top-250 Logistic Regression 与 MultiView 配对比较
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon


BASELINE_PATH = Path(
    "./result/1+2_vs_3_lambda_001_formal/"
    "metrics_per_repeat.csv"
)

TOP250_PATH = Path(
    "./result/logistic_refinement/"
    "fixed_configuration_per_repeat.csv"
)

OUTPUT_PATH = Path(
    "./result/logistic_refinement/"
    "top250_logistic_vs_multiview_paired.csv"
)


if not BASELINE_PATH.exists():
    raise FileNotFoundError(
        f"找不到 MultiView 基线结果：{BASELINE_PATH.resolve()}"
    )

if not TOP250_PATH.exists():
    raise FileNotFoundError(
        f"找不到固定参数结果：{TOP250_PATH.resolve()}"
    )


baseline_df = pd.read_csv(BASELINE_PATH)

fixed_df = pd.read_csv(TOP250_PATH)


top250_df = fixed_df[
    fixed_df["Config"] == "B_Top250_L2_balanced"
].copy()


metrics = [
    "Accuracy",
    "F1",
    "AUC",
    "Sensitivity",
    "Specificity",
]


required_columns = [
    "Repeat",
    *metrics,
]


for column in required_columns:
    if column not in baseline_df.columns:
        raise KeyError(
            f"MultiView 文件缺少列：{column}"
        )

    if column not in top250_df.columns:
        raise KeyError(
            f"Top-250 文件缺少列：{column}"
        )


paired_df = pd.merge(
    baseline_df[required_columns],
    top250_df[required_columns],
    on="Repeat",
    suffixes=(
        "_MultiView",
        "_Top250_Logistic",
    ),
)


print("成功配对次数：", len(paired_df))

if len(paired_df) != 30:
    print("警告：配对次数不是30，请检查 Repeat 列。")


comparison_rows = []


for metric in metrics:

    multiview_values = paired_df[
        f"{metric}_MultiView"
    ].to_numpy(dtype=float)

    logistic_values = paired_df[
        f"{metric}_Top250_Logistic"
    ].to_numpy(dtype=float)

    difference = (
        logistic_values
        - multiview_values
    )

    try:
        statistic, p_value = wilcoxon(
            logistic_values,
            multiview_values,
            alternative="two-sided",
            zero_method="wilcox",
        )

    except ValueError:
        statistic = np.nan
        p_value = np.nan


    comparison_rows.append(
        {
            "Metric": metric,

            "MultiView_Mean":
                multiview_values.mean(),

            "Top250_Logistic_Mean":
                logistic_values.mean(),

            "Logistic_Minus_MultiView":
                difference.mean(),

            "Wilcoxon_Statistic":
                statistic,

            "P_Value":
                p_value,
        }
    )


final_comparison_df = pd.DataFrame(
    comparison_rows
)


final_comparison_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)


display(final_comparison_df)

print(
    "结果已保存：",
    OUTPUT_PATH.resolve(),
)

成功配对次数： 30


,Metric,MultiView_Mean,Top250_Logistic_Mean,Logistic_Minus_MultiView,Wilcoxon_Statistic,P_Value
0,Accuracy,0.643750,0.697917,0.054167,89.5,0.048509
1,F1,0.516831,0.594311,0.077480,77.0,0.004115
2,AUC,0.674170,0.774459,0.100289,76.0,0.001286
3,Sensitivity,0.575758,0.660606,0.084848,100.0,0.054570
4,Specificity,0.679365,0.717460,0.038095,137.0,0.710149


结果已保存： /home/liuxy/a-projects/BPD_jj/result/logistic_refinement/top250_logistic_vs_multiview_paired.csv
